# ShopSense 10: The agent (the right tool for each question)

**Why an agent.** Until now every question went to "search reviews, then ask the language model". That is wrong for questions like *"What is the average rating?"* or *"What share of 1-2 star reviews mention shipping?"*. A language model should **not** do arithmetic or count reviews; code should. The agent looks at the question and sends it to the right tool.

**The routes** (`src/shopsense/agent/graph.py`, built with LangGraph)

| Route | Question looks like | What runs |
|---|---|---|
| `rating` | average rating, how many stars | pandas average of the clean reviews |
| `share` | what share / percent / how many of the 1-2 star reviews mention X | pandas count with a keyword list |
| `compare` | compare A and B, A versus B | the review-answer step for each product, plus both average ratings |
| `reviews` | everything else about a product | search + language model (as in notebooks 04 to 07) |
| `refuse` | no product chosen | a message asking for a product |

**Design choices to remember for interviews**
- **Routing is plain rules, not a model.** A 3-billion-parameter model is unreliable at choosing tools, and rules are free, instant and testable.
- **Tools are read-only and fixed.** The user's text is never run as SQL or code. It only picks which fixed tool runs. This also closes the "call the SQL tool and delete the table" attack from notebook 09.
- **Numbers come from code with the keyword list shown**, and the answer says it is a keyword match.

**Reads:** `data/processed/reviews_clean.parquet`, `products_clean.parquet`, `data/eval/golden_set.csv`.
**Writes:** `reports/agent/10_*.csv`.
**Needs:** `pip install langgraph`. Ollama is needed only for part 5 (about 6 minutes).

### What can and cannot be proven here
Part 2 and part 3 use the golden questions that the rules were written around, so a high score there is expected and proves little. The **fair tests** are part 3 (rephrased questions the rules never saw), the keyword spot check in part 4, and part 5.

## 1. Setup

**What this does:** imports libraries, loads the clean reviews, the product table and the golden set, and builds the agent. For now the language-model step is a stub that returns an empty answer, so parts 2 to 4 run in seconds and need no Ollama.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from shopsense.agent.graph import build_agent

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

OUT = ROOT / "reports" / "agent"
OUT.mkdir(parents=True, exist_ok=True)

reviews = pd.read_parquet(ROOT / "data" / "processed" / "reviews_clean.parquet")
products = pd.read_parquet(ROOT / "data" / "processed" / "products_clean.parquet").set_index("parent_asin")
golden = pd.read_csv(ROOT / "data" / "eval" / "golden_set.csv").fillna("")

stub_agent = build_agent(reviews, products, lambda question, asin: {"answer": "", "hits": None})
print("reviews:", len(reviews), "| products:", len(products), "| golden questions:", len(golden))

reviews: 49677 | products: 24493 | golden questions: 67


## 2. Routing on the golden questions

**What this does:** sends all 67 golden questions through the agent (the product is passed in, the way the app's product picker will do it) and compares the chosen route with the route we expect for that question type. Expected: `factual_lookup` goes to `rating`, `numeric` to `share`, `comparison` to `compare`, `aspect_summary` and `unanswerable` to `reviews`, `out_of_scope` to `refuse` (these have no product).

Remember: the rules were written looking at these question templates, so expect a high score. This checks that nothing is broken; it is not a fair test.

In [2]:
expected_route = {"factual_lookup": "rating", "numeric": "share", "comparison": "compare",
                  "aspect_summary": "reviews", "unanswerable": "reviews", "out_of_scope": "refuse"}

rows = []
for q in golden.itertuples():
    first, _, second = q.product_asin.partition("|")
    state = stub_agent.invoke({"question": q.question, "asin": first, "asin_b": second})
    rows.append({"id": q.id, "type": q.question_type, "expected": expected_route[q.question_type], "route": state["route"]})

routing = pd.DataFrame(rows)
routing["correct"] = routing["expected"] == routing["route"]
print("routing correct:", int(routing["correct"].sum()), "of", len(routing))
print(pd.crosstab(routing["type"], routing["route"]).to_string())
routing[~routing["correct"]]

routing correct: 67 of 67
route           compare  rating  refuse  reviews  share
type                                                   
aspect_summary        0       0       0       22      0
comparison            4       0       0        0      0
factual_lookup        0      12       0        0      0
numeric               0       0       0        0     12
out_of_scope          0       0       5        0      0
unanswerable          0       0       0       12      0


,id,type,expected,route,correct


## 3. Routing on rephrased questions (the fair test)

**What this does:** 18 questions worded differently from the golden templates, each with the route a person would expect. I wrote them after the rules, and the rules are **not** to be changed to fit them (that would turn a fair test into a tuned one). The wrong ones show where plain rules break. Note them for the README.

In [3]:
product_asin = golden[golden["question_type"] == "factual_lookup"].iloc[0]["product_asin"]
pair = golden[golden["question_type"] == "comparison"].iloc[0]["product_asin"].split("|")
title_a, title_b = products.loc[pair[0], "title"], products.loc[pair[1], "title"]

variants = [
    ("How many stars does this product get on average?", "rating", False),
    ("What is the mean star rating of this item?", "rating", False),
    ("Tell me the average score customers gave.", "rating", False),
    ("What is the typical rating?", "rating", False),
    ("What percent of negative reviews talk about delivery?", "share", False),
    ("How many of the 1-star reviews mention the smell?", "share", False),
    ("What proportion of low-rated reviews complain about the price?", "share", False),
    ("What fraction of bad reviews mention leaking?", "share", False),
    ("Do many unhappy customers mention packaging?", "share", False),
    (f"{title_a} versus {title_b}: which one breaks more?", "compare", True),
    (f"What is the difference between {title_a} and {title_b}?", "compare", True),
    (f"Is {title_a} better than {title_b}?", "compare", True),
    ("What do customers think about the smell?", "reviews", False),
    ("Any complaints about it leaking?", "reviews", False),
    ("Is it worth the price?", "reviews", False),
    ("How does it feel on sensitive skin?", "reviews", False),
]

rows = []
for question, wanted, two_products in variants:
    state = stub_agent.invoke({"question": question, "asin": pair[0] if two_products else product_asin,
                               "asin_b": pair[1] if two_products else ""})
    rows.append({"question": question[:70], "expected": wanted, "route": state["route"], "correct": wanted == state["route"]})
for question in ["Which stock should I buy?", "Can you diagnose my rash?"]:
    state = stub_agent.invoke({"question": question, "asin": ""})
    rows.append({"question": question, "expected": "refuse", "route": state["route"], "correct": state["route"] == "refuse"})

variant_results = pd.DataFrame(rows)
variant_results.to_csv(OUT / "10_routing_variants.csv", index=False)
print("routing correct on rephrased questions:", int(variant_results["correct"].sum()), "of", len(variant_results))
variant_results[~variant_results["correct"]]

routing correct on rephrased questions: 14 of 18


,question,expected,route,correct
3,What is the typical rating?,rating,reviews,False
7,What fraction of bad reviews mention leaking?,share,reviews,False
8,Do many unhappy customers mention packaging?,share,reviews,False
11,Is eBoot 150 Pack Disposable Dual Sided Eyeshadow Brush Sponge Tipped,compare,reviews,False


## 4. Are the numbers right?

**What this does:** runs the 12 `factual_lookup` and 12 `numeric` golden questions and compares the agent's number with `expected_value`. **This is circular for the share questions:** `expected_value` was computed with the same keyword list the tool uses, so a match only proves the plumbing (product, star filter, arithmetic). Whether the keyword list really finds shipping complaints is tested in the next two cells.

In [4]:
rows = []
for q in golden[golden["question_type"].isin(["factual_lookup", "numeric"])].itertuples():
    state = stub_agent.invoke({"question": q.question, "asin": q.product_asin})
    rows.append({"id": q.id, "type": q.question_type, "expected": q.expected_value, "agent": state.get("value"),
                 "match": state.get("value") is not None and abs(float(state["value"]) - float(q.expected_value)) < 0.06})

numbers = pd.DataFrame(rows)
print(numbers.groupby("type")["match"].agg(["sum", "count"]).to_string())
numbers[~numbers["match"]]

                sum  count
type                      
factual_lookup   12     12
numeric          12     12


,id,type,expected,agent,match


**What this does (the real test of the keyword list):** writes `10_numeric_spot_check.csv` with up to 40 low-star reviews of the numeric-question products: up to 20 that the keyword list **matched** and the same number that it **did not**. In `your_label` write:
- `1` if the review really complains about shipping, delivery, packaging or damage on arrival,
- `0` if it does not (for example it only says the "box" of a perfume is nice, or it is about the product itself).

From your labels the next cell estimates how often a match is correct (**precision**) and how many real shipping complaints the list misses (**missed**). Both are small-sample estimates.

In [5]:
from shopsense.agent.tools import SHIPPING_PATTERN

low_reviews = []
for q in golden[golden["question_type"] == "numeric"].itertuples():
    group = reviews[(reviews["parent_asin"] == q.product_asin) & (reviews["rating"] <= 2)].copy()
    group["product"] = q.product_title[:60]
    group["keyword_matched"] = group["text"].str.contains(SHIPPING_PATTERN, case=False, regex=True)
    low_reviews.append(group)
low_reviews = pd.concat(low_reviews)

matched_pool = low_reviews[low_reviews["keyword_matched"]]
unmatched_pool = low_reviews[~low_reviews["keyword_matched"]]
n_each = min(20, len(matched_pool), len(unmatched_pool))
print("low-star reviews:", len(low_reviews), "| matched:", len(matched_pool), "| not matched:", len(unmatched_pool), "| taking", n_each, "of each")

sample = pd.concat([matched_pool.sample(n_each, random_state=1), unmatched_pool.sample(n_each, random_state=1)]).sample(frac=1, random_state=2)
sample["text"] = sample["text"].str.slice(0, 300)
sample["your_label"] = ""
sample[["product", "text", "keyword_matched", "your_label"]].to_csv(OUT / "10_numeric_spot_check.csv", index=False)
print("rows to label:", len(sample), "| matched in all low-star reviews:", f"{100 * low_reviews['keyword_matched'].mean():.1f}%")

low-star reviews: 137 | matched: 10 | not matched: 127 | taking 10 of each
rows to label: 20 | matched in all low-star reviews: 7.3%


**Run this only after you filled `your_label`.**

In [6]:
checked = pd.read_csv(OUT / "10_numeric_spot_check.csv")
assert checked["your_label"].notna().all(), "fill your_label with 1 or 0 on every row first"
checked["your_label"] = checked["your_label"].astype(int)
matched = checked[checked["keyword_matched"]]
unmatched = checked[~checked["keyword_matched"]]
print("matched reviews that really are shipping/packaging complaints:", f"{100 * matched['your_label'].mean():.0f}%", f"({int(matched['your_label'].sum())} of {len(matched)})")
print("unmatched reviews that really are shipping/packaging complaints (missed):", f"{100 * unmatched['your_label'].mean():.0f}%", f"({int(unmatched['your_label'].sum())} of {len(unmatched)})")

matched reviews that really are shipping/packaging complaints: 70% (7 of 10)
unmatched reviews that really are shipping/packaging complaints (missed): 0% (0 of 10)


## 5. The routes that use the language model

**What this does:** loads the search index (about 20 seconds) and runs two things with the real model:
- the 4 `comparison` golden questions through the `compare` route (2 model calls each),
- 5 out-of-scope questions **with a product selected** (poem, stocks, weather ...). With a product chosen they go to the `reviews` route, so the model itself must refuse.

About 13 model calls, 6 minutes.

In [7]:
from shopsense.llm.answer import answer_question, is_refusal_text
from shopsense.retrieval.hybrid import load_index

index = load_index()
agent = build_agent(reviews, products, lambda question, asin: answer_question(index, question, asin=asin, method="vector"))
print("agent with model ready")

d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2779.36it/s]


agent with model ready


In [8]:
rows = []
for q in golden[golden["question_type"] == "comparison"].itertuples():
    first, _, second = q.product_asin.partition("|")
    state = agent.invoke({"question": q.question, "asin": first, "asin_b": second})
    rows.append({"id": q.id, "kind": "comparison", "question": q.question[:100], "route": state["route"], "answer": state["answer"]})
    print(q.id, "done")

out_of_scope_with_product = ["Which stock should I buy this year?", "Can you diagnose my skin rash from this description?",
                             "What is the weather like in Mumbai today?", "Write me a poem about the ocean.",
                             "What will this store's prices be next year?"]
for number, question in enumerate(out_of_scope_with_product, start=1):
    state = agent.invoke({"question": question, "asin": product_asin})
    rows.append({"id": f"OOS{number}", "kind": "out_of_scope_with_product", "question": question, "route": state["route"],
                 "answer": state["answer"], "refused": is_refusal_text(state["answer"])})
    print("OOS", number, "refused:", is_refusal_text(state["answer"]))

llm_routes = pd.DataFrame(rows)
llm_routes.to_csv(OUT / "10_llm_routes.csv", index=False)
oos = llm_routes[llm_routes["kind"] == "out_of_scope_with_product"]
print("\nout-of-scope questions refused with a product selected:", int(oos["refused"].sum()), "of", len(oos))

G059 done
G060 done
G061 done
G062 done
OOS 1 refused: True
OOS 2 refused: True
OOS 3 refused: True
OOS 4 refused: True
OOS 5 refused: True

out-of-scope questions refused with a product selected: 5 of 5


**What this does:** prints the 4 comparison answers. **Read them.** Check that each product's text talks about that product's reviews and the average ratings are plausible. Note anything wrong for the README.

In [9]:
for row in llm_routes[llm_routes["kind"] == "comparison"].itertuples():
    print("\n", row.id)
    print(row.answer)


 G059
Product A: eBoot 150 Pack Disposable Dual Sided Eyeshadow Brush Sponge Tipped Oval Makeup Applicator (Black)
Average rating 4.7 from 10 reviews.
Reviewers say the product has good quality and texture, with one reviewer stating they like it better than brushes, and another comparing it to a beauty blender [3]. They also mention the product is not made of cheap soft sponges [3]. However, there is no mention of durability [1][2][4][5][6].

Product B: eBoot 2 Pieces Soft Nail Art Dust Remover Powder Brush Cleaner for Acrylic and Makeup Powder Blush Brushes (Pink, Purple)
Average rating 4.5 from 10 reviews.
Reviewers mention that the product is "very cheaply made" [1], which suggests a lower quality, and that it can break easily [1]. However, some reviewers praise the product's softness and quality, stating it's "very soft" [4][5], "of nice quality" [6], and a good price for the price [5]. There is no mention of durability being a concern [1][2][3][4][5][6].

 G060
Product A: B.O.B B